# MiniGPT-Stack pretraining launcher (THIN — all logic lives in train/)

One Save Version = one MODE. Set MODE in the next cell, run top to bottom, Save Version.
- `prep`: full tokenizer-data prepare once (streams ~2GB). Output: shards dataset.
- `calibrate`: throughput probe + fp16 gate + kill-resume drill + LR sweep. Output: numbers.
- `main`: resume-latest training, clean exit before the kill. Output: checkpoints.

In [ ]:
MODE = "prep"  # prep | calibrate | main  <-- SET THIS, then run all
assert MODE in ("prep", "calibrate", "main"), MODE
import torch
print("MODE", MODE)
print("gpu", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| cuda build", torch.version.cuda,
      "| cuda avail", torch.cuda.is_available())
assert torch.cuda.is_available(), "enable GPU in notebook settings"
# Copy these three lines into docs/decisions.md with the run results.

In [ ]:
import os
REPO = "https://github.com/Priyanshaur/Glassbox.git"
ROOT = "/kaggle/working/Glassbox"
if not os.path.exists(ROOT):
    !git clone $REPO $ROOT
else:
    !git -C $ROOT pull --ff-only
%cd $ROOT
!git rev-parse --short HEAD

In [ ]:
# Install everything EXCEPT torch: Kaggle's preinstalled torch stays (see decisions.md).
!grep -v '^torch' requirements.txt | pip install -r /dev/stdin
!python -m pytest tests/test_config.py tests/test_model.py tests/test_pretrain.py -q 2>&1 | tail -n 2

## Restore previous checkpoints (calibrate/main only)
Attach the previous version's output first (Add Data -> previous run output).

In [ ]:
import glob, os, shutil
if MODE not in ("calibrate", "main"):
    print("skip: prep starts fresh")
else:
    cands = sorted(glob.glob("/kaggle/input/*/checkpoints/*/*.pt"))
    if cands:
        src_dir = os.path.dirname(cands[-1])
        dst_dir = os.path.join("checkpoints", os.path.basename(src_dir))
        os.makedirs(dst_dir, exist_ok=True)
        for f in glob.glob(os.path.join(src_dir, "*")):
            if os.path.basename(f) not in os.listdir(dst_dir):
                shutil.copy(f, dst_dir)
        print("restored", len(os.listdir(dst_dir)), "files ->", dst_dir)
        print("newest:", max(glob.glob(os.path.join(dst_dir, 'ckpt_step*.pt'))))
    else:
        print("no previous checkpoints attached; starting fresh")

## Shards
prep: run the FULL prepare once. calibrate/main: attach the shards dataset and print exact counts.

In [ ]:
import glob, json, os
if MODE == "prep":
    !python -m data.prepare --tokenizer tokenizer/tok8192.json --out_dir data
else:
    hit = glob.glob("/kaggle/input/*tinystories*/data/meta.json") + \
          glob.glob("/kaggle/input/*/meta.json")
    assert hit, "attach the shards dataset first"
    shard_dir = os.path.dirname(hit[0])
    if not os.path.exists("data/meta.json"):
        os.symlink(shard_dir, "data") if not os.path.lexists("data") else None
    meta = json.load(open("data/meta.json"))
    tr = sum(s["tokens"] for s in meta["shards"] if s["split"] == "train")
    va = sum(s["tokens"] for s in meta["shards"] if s["split"] == "validation")
    print(f"EXACT counts from data/meta.json: train={tr} val={va} vocab={meta['vocab_size']}")
    for s in meta["shards"]:
        mb = os.path.getsize(os.path.join("data", s["file"])) / 1024 / 1024
        print(f"  {s['file']} {s['tokens']} toks {mb:.1f}MB")
    TRAIN_TOKS, VAL_TOKS = tr, va

## Throughput + memory probe (calibrate only)
60 steps at micro_batch_seqs 8/16/32. Peak GPU MB is polled from nvidia-smi in a thread
(the trainer runs in a subprocess, so torch peak-stats would miss it).

In [ ]:
import subprocess, threading, time
PROBE = {}
if MODE != "calibrate":
    print("skip: probe runs in calibrate mode")
else:
    def _poll(stop, rec):
        mx = 0
        while not stop.is_set():
            try:
                q = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=memory.used",
                     "--format=csv,noheader,nounits"], text=True)
                mx = max(mx, int(q.strip().split()[0]))
            except Exception:
                pass
            time.sleep(2)
        rec.append(mx)
    for MB in (8, 16, 32):
        stop, rec = threading.Event(), []
        th = threading.Thread(target=_poll, args=(stop, rec))
        th.start()
        r = subprocess.run(
            ["python", "-m", "train.pretrain", "--config", "configs/base_30m.yaml",
             "--resume", "none", "--micro_batch_seqs", str(MB),
             "--max_steps", "60", "--ckpt_dir", f"/tmp/probe_{MB}"],
            capture_output=True, text=True)
        stop.set(); th.join()
        print(r.stdout[-800:] if r.returncode == 0 else r.stderr[-2000:])
        PROBE[MB] = rec[0] if rec else -1
        print(f"micro_batch {MB}: peak GPU mem ~= {PROBE[MB]} MB")

## Shakedown summary (calibrate only): tok/s table + skip-rate gate (fail >= 1%)

In [ ]:
import json
PROBE_TOKS = None
if MODE != "calibrate":
    print("skip: summary runs in calibrate mode")
else:
    best, rows = 0, []
    for MB in (8, 16, 32):
        recs = [json.loads(l) for l in open(f"/tmp/probe_{MB}/metrics.jsonl")]
        toks = sum(r["tok_s"] for r in recs) / len(recs)
        rate = recs[-1]["skips"] / 60  # skips are cumulative; 60 probe steps
        gate = "PASS" if rate < 0.01 else "FAIL"
        rows.append((MB, toks, PROBE[MB], rate, gate))
        print(f"mb={MB:2d} tok/s={toks:7.0f} peak_mem={PROBE[MB]}MB skip_rate={rate:.4f} [{gate}]")
    PROBE_TOKS = max(r[1] for r in rows if r[4] == "PASS")
    print(f"PROBE_TOKS (best passing) = {PROBE_TOKS:.0f}")
    assert PROBE_TOKS > 0, "no probe passed the skip gate — lower LR before burning quota"

## Budget (calibrate + main)
Computes max_steps from MEASURED tok/s and a 5.5h target — replaces the provisional default.
In main mode, paste PROBE_TOKS from the calibrate output.

In [ ]:
TOK_S = globals().get("PROBE_TOKS") or None  # calibrate fills this; main: paste the number
# TOK_S = 80000  # <-- main mode: uncomment + paste calibrate's PROBE_TOKS if the cell above is empty
TARGET_H, BATCH = 5.5, 65536
assert TOK_S, "need measured tok/s (run calibrate first)"
budget = TOK_S * TARGET_H * 3600
passes = budget / TRAIN_TOKS
MAX_STEPS = int(budget // BATCH)
print(f"train_tokens={TRAIN_TOKS} val_tokens={VAL_TOKS}")
print(f"tok/s={TOK_S:.0f} target={TARGET_H}h -> budget={budget/1e9:.2f}B tokens")
print(f"passes over train data ~= {passes:.2f} -> MAX_STEPS={MAX_STEPS}")

## Kill-resume drill + LR sweep (calibrate only)
Drill: 150 steps, clean stop, resume to 300, confirm metrics.jsonl continues with no loss jump.
Sweep: 3 values x 900 steps, same schedule, compare val at the same step; prefer lower LR on ties.

In [ ]:
if MODE != "calibrate":
    print("skip: drill+sweep run in calibrate mode")
else:
    !python -m train.pretrain --config configs/base_30m.yaml --resume none --max_steps 150 2>&1 | tail -n 3
    !python -m train.pretrain --config configs/base_30m.yaml --resume latest --max_steps 300 2>&1 | tail -n 3
    !tail -n 6 checkpoints/base_30m/metrics.jsonl
    print("check: steps 0..299 continuous, no loss jump at 150")

In [ ]:
if MODE != "calibrate":
    print("skip: sweep runs in calibrate mode")
else:
    import subprocess
    sweep = {}
    for LR in ("3e-4", "6e-4", "1e-3"):
        subprocess.run(["python", "-m", "train.pretrain", "--config",
                        "configs/base_30m.yaml", "--resume", "none", "--lr", LR,
                        "--max_steps", "900", "--ckpt_dir", f"/tmp/sweep_{LR}"],
                       check=True)
        last = [l for l in open(f"/tmp/sweep_{LR}/metrics.jsonl") if '"val"' in l][-1]
        v = __import__("json").loads(last)["val"]
        sweep[LR] = v
        print(f"lr={LR} val={v:.4f}")
    print("--- sweep val losses ---")
    for LR, v in sweep.items():
        print(f"  lr={LR}: val={v:.4f}")
    best = min(sweep, key=sweep.get)
    # Close = within 2% of best -> prefer the LOWEST lr (stability over speed).
    close = [LR for LR in sorted(sweep, key=float)
             if (sweep[LR] - sweep[best]) / sweep[best] < 0.02]
    proposed = close[0]
    print(f"PROPOSED_LR = {proposed} (NOT applied — confirm, then paste into the main cell)")

## Main run (main only): resume latest, computed max_steps, clean exit. Then Save Version.

In [ ]:
if MODE != "main":
    print("skip: main run in main mode")
else:
    import subprocess
    CONFIRMED_LR = None  # paste the confirmed sweep LR here; None = config default
    argv = ["python", "-m", "train.pretrain", "--config", "configs/base_30m.yaml",
            "--resume", "latest", "--max_hours", "8", "--max_steps", str(MAX_STEPS)]
    if CONFIRMED_LR:
        argv += ["--lr", str(CONFIRMED_LR)]
    print("LR:", CONFIRMED_LR or "config default (6e-4)")
    subprocess.run(argv, check=False)

## Backup (main only): newest checkpoints -> private HF repo, graceful skip.
HF_TOKEN goes in Kaggle Secrets. Weights-only copies ride along on Save Version.

In [ ]:
import glob, os
if MODE != "main":
    print("skip: backup runs in main mode")
else:
    from kaggle_secrets import UserSecretsClient
    try:
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        tok = None
        print("no HF_TOKEN secret; skipping backup:", e)
    ckpts = sorted(glob.glob("checkpoints/*/ckpt_step*.pt"))
    if tok and ckpts:
        try:
            import importlib
            try:
                hhub = importlib.import_module("huggingface_hub")
            except ImportError:
                !pip install -q huggingface_hub
                hhub = importlib.import_module("huggingface_hub")
            hhub.login(token=tok)
            api = hhub.HfApi()
            repo = api.create_repo("minigpt-stack-ckpts", private=True, exist_ok=True).repo_id
            for f in ckpts[-2:]:
                api.upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f), repo_id=repo)
            print("backed up to", repo, ckpts[-2:])
        except Exception as e:
            print("backup skipped (retry next session):", e)
    elif not ckpts:
        print("no checkpoints to back up")